In [1]:
import pandas as pd

df = pd.read_csv("train.csv")
df.head()


,text,label,dataset
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive,urun_yorumlari
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive,urun_yorumlari
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive,urun_yorumlari
3,Çünkü aranan tapınak bu bölgededir .,Notr,wiki
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive,urun_yorumlari


In [2]:
df = df[["text", "label"]]
df.head()


,text,label
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive
3,Çünkü aranan tapınak bu bölgededir .,Notr
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive


In [3]:
df = df[df["label"].isin(["Positive", "Negative", "Notr"])]
df.head()


,text,label
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive
3,Çünkü aranan tapınak bu bölgededir .,Notr
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive


In [4]:
df = df.dropna()
df = df[df["text"].str.len() > 5]
df.head()


,text,label
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive
3,Çünkü aranan tapınak bu bölgededir .,Notr
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive


In [7]:
import re
import emoji

def clean_text(text):
    text = str(text).lower()

    # linkleri sil
    text = re.sub(r"http\S+|www\S+", "", text)

    # mention sil
    text = re.sub(r"@\w+", "", text)

    # hashtag sil
    text = re.sub(r"#\w+", "", text)

    # emoji sil
    text = emoji.replace_emoji(text, replace="")

    # özel karakterleri sadeleştir
    text = re.sub(r"[^a-zA-Z0-9çğıöşüÇĞİÖŞÜ\s.,!?]", " ", text)

    # çoklu boşlukları tek boşluk yap
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [8]:
df["clean_text"] = df["text"].apply(clean_text)
df.head()



,text,label,clean_text
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive,"ürünlerden çok memnunum, kesinlikle herkese ta..."
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive,"hızlı kargo, temiz alışveriş.teşekkür ederim."
3,Çünkü aranan tapınak bu bölgededir .,Notr,çünkü aranan tapınak bu bölgededir .
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive,bu telefonu başlıca alma nedenlerim ise elimde...


In [9]:
df["label"] = df["label"].replace({
    "Positive": "positive",
    "Negative": "negative",
    "Notr": "notr"
})
df["label"].value_counts()


label
positive    235812
notr        153801
negative     50875
Name: count, dtype: int64

In [10]:
min_samples = df["label"].value_counts().min()

df_positive = df[df.label == "positive"].sample(min_samples, random_state=42)
df_notr     = df[df.label == "notr"].sample(min_samples, random_state=42)
df_negative = df[df.label == "negative"].sample(min_samples, random_state=42)

df_balanced = pd.concat([df_positive, df_notr, df_negative])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced["label"].value_counts()


label
positive    50875
negative    50875
notr        50875
Name: count, dtype: int64

In [14]:
import pandas as pd
import numpy as np
import re
import emoji
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pickle


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced["clean_text"],
    df_balanced["label"],
    test_size=0.2,
    random_state=42
)

len(X_train), len(X_test)


(122100, 30525)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=120000,     # Büyük dataset için ideal
    ngram_range=(1, 2),      # unigram + bigram
    sublinear_tf=True,       # tf-log scaling daha stabil
    dtype=np.float32         # GPU/CPU kullanımını optimize eder
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

X_train_vec.shape, X_test_vec.shape


((122100, 120000), (30525, 120000))

In [16]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        n_jobs=-1,
        verbose=1
    ),

    "sgd_logistic": SGDClassifier(
        loss="log_loss",    # Logistic Regression vari
        max_iter=10,
        tol=None,
        random_state=42
    ),

    "naive_bayes": MultinomialNB()
}

results = {}

for name, clf in models.items():
    print("=" * 80)
    print(f"Training model: {name}")
    clf.fit(X_train_vec, y_train)

    y_pred = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)

    print(f"\nAccuracy ({name}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    results[name] = {
        "model": clf,
        "accuracy": acc
    }

print("\nSUMMARY:")
for name, result in results.items():
    print(f"{name}: acc = {result['accuracy']:.4f}")


Training model: logistic_regression


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.



Accuracy (logistic_regression): 0.9160

Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.89      0.90     10351
        notr       0.94      0.98      0.96     10035
    positive       0.90      0.88      0.89     10139

    accuracy                           0.92     30525
   macro avg       0.92      0.92      0.92     30525
weighted avg       0.92      0.92      0.92     30525


Confusion Matrix:
[[9185  284  882]
 [ 121 9847   67]
 [ 856  353 8930]]
Training model: sgd_logistic

Accuracy (sgd_logistic): 0.8645

Classification Report:
              precision    recall  f1-score   support

    negative       0.87      0.81      0.84     10351
        notr       0.85      0.98      0.91     10035
    positive       0.88      0.81      0.84     10139

    accuracy                           0.86     30525
   macro avg       0.87      0.87      0.86     30525
weighted avg       0.87      0.86      0.86     30525


Confusion Mat

In [17]:
from sklearn.metrics import accuracy_score

train_pred = results["logistic_regression"]["model"].predict(X_train_vec)
train_acc = accuracy_score(y_train, train_pred)

print("Train accuracy:", train_acc)
print("Test accuracy:", 0.9160)


Train accuracy: 0.9527682227682228
Test accuracy: 0.916


In [18]:
best_name = max(results, key=lambda k: results[k]["accuracy"])
best_model = results[best_name]["model"]
best_acc = results[best_name]["accuracy"]

print(f"En iyi model: {best_name} | Accuracy: {best_acc:.4f}")


En iyi model: logistic_regression | Accuracy: 0.9160


In [19]:
import pickle

# Modeli kaydet
with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

# TF-IDF vectorizer'ı kaydet
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("sentiment_model.pkl ve tfidf_vectorizer.pkl başarıyla kaydedildi!")


sentiment_model.pkl ve tfidf_vectorizer.pkl başarıyla kaydedildi!
